# PriX (JAX) vs. CuRL (PyTorch) — GPU Benchmark on Google Colab

Make sure **Runtime → Change runtime type → Hardware accelerator** is set to **T4 GPU** (or **L4 / A100 GPU**).

In [ ]:
!nvidia-smi
!pip install -q "jax[cuda12]" pywavelets pyyaml onnx onnxruntime pandas
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
import jax
print("JAX devices:", jax.devices())

In [ ]:
# Clone PriX and CuRL repositories
!git clone https://github.com/mugurbil/prix.git /content/prix
!git clone https://github.com/jimouris/curl.git /content/curl
!pip install -q -e /content/prix

In [ ]:
# Run PriX GPU LLM benchmarks (Wavelet LUT mode & Fission Evaluator mode)
!python /content/prix/examples/llms/launcher.py --world_size 2 --multiprocess --model BertTiny --tensor_size 1,16 --not-full --device cuda
!python /content/prix/examples/llms/launcher.py --world_size 2 --multiprocess --model GPT2 --tensor_size 1,16 --not-full --device cuda
!python /content/prix/examples/llms/launcher.py --world_size 2 --evaluator_size 1 --multiprocess --model GPT2 --tensor_size 1,16 --not-full --device cuda

In [ ]:
# Side-by-side PriX (JAX GPU) vs. CuRL (PyTorch GPU) Benchmark
import sys
sys.path.insert(0, "/content/prix")
sys.path.insert(0, "/content/curl")

import time
import numpy as np
import torch
import jax
import jax.numpy as jnp

import curl as curl_pkg
import curl.communicator as curl_comm
from curl.config import cfg as curl_cfg
import prix
import prix.communicator as prix_comm
from prix.config import cfg as prix_cfg
from examples.llms.models import (
    BertTiny as PrixBertTiny,
    BertBase as PrixBertBase,
    GPT2 as PrixGPT2,
    Llama as PrixLlama,
    ModernBert as PrixModernBert,
)
from examples.llm.models import (
    BertTiny as CurlBertTiny,
    BertBase as CurlBertBase,
    GPT2 as CurlGPT2,
    Llama as CurlLlama,
    ModernBert as CurlModernBert,
)

print("JAX devices:", jax.devices())
print("PyTorch CUDA available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
